# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR⁲ dataset using the `mlcroissant` library. We will use the Croissant schema definition to dynamically discover and analyze the dataset's structure, fields, and records for effective and reproducible analysis.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load and access metadata and records from the FAIR⁲ dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(url)

# Access the dataset metadata
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', None))
print("Authors:",[author for author in getattr(metadata, 'author', [])])
print("Keywords:", getattr(metadata, 'keywords', None))

## 2. Data Overview
Review available record sets, fields, and their IDs. This allows us to understand what tables (record sets) exist in the dataset and which fields/columns are available in each. All entities are referenced by their `@id` fields per FAIR and Croissant best practices.

In [ ]:
# List all available record sets by their @id
record_sets = dataset.record_sets
print(f"Number of record sets found: {len(record_sets)}\n")

for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs.id}")
    print(f"    Name: {rs.name}")
    print(f"    Description: {getattr(rs, 'description', 'N/A')}")
    # List all fields within this recordset
    fields = getattr(rs, 'fields', [])
    for field in fields:
        print(f"        Field @id: {field.id} | name: {field.name} | dataType: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Choose record set and field `@id`s as revealed above. All entity references use `@id`.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for the given record set @id
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'RecordSet {record_set_id}: Loaded {len(records)} records. Columns: {dataframes[record_set_id].columns.tolist()}')
    else:
        print(f'RecordSet {record_set_id} contains NO records or is metadata only.')

# For demonstration, pick first record set with data as the main table
main_record_set_id = None
for _id in record_set_ids:
    if _id in dataframes:
        main_record_set_id = _id
        break
if main_record_set_id:
    print(f'\nUsing main RecordSet @id for further analysis: {main_record_set_id}')
    print(dataframes[main_record_set_id].head())
else:
    print('No record sets with data found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalization, and grouping. Always refer to fields by their `@id`.

You may need to inspect real field names in the data shown above to select meaningful numeric and categorical fields for your analysis.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print('Numeric columns:', df.select_dtypes('number').columns.tolist())
    if len(df.select_dtypes('number').columns) > 0:
        numeric_field = df.select_dtypes('number').columns[0]  # Use first numeric field
        print(f'Using numeric field for demonstration: {numeric_field}')
        # Simple EDA: filter records, normalize, and group
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f'Filtered records (where {numeric_field} > mean): {len(filtered_df)} rows')
        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field
        possible_groups = [col for col in df.columns if df[col].nunique() < 32 and df[col].dtype==object]
        group_field = possible_groups[0] if possible_groups else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (first rows):")
            print(grouped_df.head())
        else:
            print("No suitable categorical grouping field found.")
    else:
        print('No numeric fields found for EDA.')
else:
    print('No record set with data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields. Here we demonstrate a histogram and a boxplot for the main numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and len(df.select_dtypes('number').columns) > 0:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    df[numeric_field].hist(bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric data to visualize.")

## 6. Conclusion
In this notebook, we loaded the FAIR⁲ dataset using `mlcroissant`, explored the available record sets and fields using their Croissant `@id`s, and performed basic exploratory data analysis and visualizations. Croissant makes it easy to programmatically discover and utilize dataset structure for robust data science workflows.